# 1.3 TCN 그리드 서치

- **그리드** · 시간창 4초·20초 × 24·48채널 × AdamW·Adam, 8조합.
- **고정** · 학습률 0.002, 가중치 감쇠 0.001, 최대 20 epoch, 내부 검증 3회 연속 미개선 시 조기 종료.
- **평가** · 바깥 DFL 경기 1개를 뺀다. 나머지 6경기 중 지정한 내부 검증 경기로 후보 조합과 epoch를 선택한다. 선택된 설정으로 6경기 재학습 후 바깥 경기를 평가한다.
- **출력** · 모델별 평균과 경기별 결과, 내부 검증으로 선택한 조합의 바깥 점수. 전체 바깥 평균 최고 조합은 비교용 집계다.
- **실행** · 같은 폴더의 `dfl_tcn_tuning.ipynb`와 `dfl_model_comparison.ipynb`를 유지한 채 셀을 순서대로 실행한다. 완료한 동일 조합은 이전 체크포인트에서 읽는다.

In [ ]:
from itertools import product
import json
from pathlib import Path

root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/train_dfl.py").is_file()
)
tuning = root / "model/영상추출/모델/dead_ball/dfl_tcn_tuning.ipynb"
tuning_cells = {cell["id"]: cell for cell in json.loads(tuning.read_text())["cells"]}
for cell_id in ("dfl13-tune-2", "dfl13-tune-4"):
    source = "".join(tuning_cells[cell_id]["source"])
    exec(compile(source, f"{tuning}:{cell_id}", "exec"))

PILOT_CANDIDATES = CANDIDATES
PILOT = OUT / "checkpoint.json"
NOTEBOOK = folder / "dfl_tcn_grid.ipynb"
RUN_ID = "dfl_tcn_grid_20261111_01"
OUT = ROOT / "model/영상추출/추출csv/dead_ball" / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)
DILATIONS = {4: (1, 2, 3, 4), 20: (1, 3, 8, 13, 25)}
GRID = {
    f"{seconds}s {width} {optimizer}": dict(window_s=seconds, width=width,
                                            optimizer=optimizer, dilations=DILATIONS[seconds])
    for seconds, width, optimizer in product((4, 20), (24, 48), ("AdamW", "Adam"))
}
assert len(GRID) == 8 and all(GRID[key] == spec for key, spec in PILOT_CANDIDATES.items())

In [ ]:
def save_checkpoint(path, fingerprint, results):
    pending = path.with_suffix(".tmp")
    pending.write_text(json.dumps(dict(fingerprints=fingerprint, results=results)), encoding="utf-8")
    pending.replace(path)


def ordered_scores(fold):
    return {row["stage"]: row for row in fold["scores"]}


def best_inner(fold):
    return max(row["inner_f1"] for row in fold["history"])


def report_grid(results, fingerprint, inherited, preparation_s):
    indexed = {(fold["model"], fold["match"]): fold for fold in results}
    expected = set(product(GRID, MATCHES))
    assert set(indexed) == expected and len(indexed) == len(results)
    flat = []
    histories = []
    for name, held in product(GRID, MATCHES):
        fold = indexed[name, held]
        assert fold["inner_match"] != held and set(ordered_scores(fold)) == {"viterbi", "line_out"}
        assert any(row["epoch"] == fold["selected_epoch"] and row["inner_f1"] == best_inner(fold)
                   for row in fold["history"])
        fields = {key: value for key, value in fold.items() if key not in ("scores", "history")}
        flat.extend(dict(fields, **score) for score in fold["scores"])
        histories.extend(dict(model=name, match=held, inner_match=fold["inner_match"], **row)
                         for row in fold["history"])
    table = pd.DataFrame(flat)
    assert len(table) == 112 and table.alive_spell_f1.between(0, 1).all()
    summary = (
        table.groupby(["model", "stage"], sort=False)
        .agg(matches=("match", "count"), alive_spell_f1=("alive_spell_f1", "mean"),
             dead_frame_f1=("dead_f1", "mean"), false_dead=("false_dead", "sum"),
             missed_dead=("missed_dead", "sum"), total_s=("total_s", "sum"),
             epoch_min=("selected_epoch", "min"), epoch_max=("selected_epoch", "max"))
        .reset_index()
    )
    choices = []
    for held in MATCHES:
        winner = max((indexed[name, held] for name in GRID), key=best_inner)
        stages = ordered_scores(winner)
        choices.append(dict(
            match=held, inner_match=winner["inner_match"], model=winner["model"],
            selected_epoch=winner["selected_epoch"], inner_f1=best_inner(winner),
            viterbi_f1=stages["viterbi"]["alive_spell_f1"], line_out_f1=stages["line_out"]["alive_spell_f1"],
            false_dead=stages["line_out"]["false_dead"], missed_dead=stages["line_out"]["missed_dead"],
        ))
    selected = pd.DataFrame(choices)
    table.to_csv(OUT / "per_match.csv", index=False)
    summary.to_csv(OUT / "model_summary.csv", index=False)
    pd.DataFrame(histories).to_csv(OUT / "inner_epoch_history.csv", index=False)
    selected.to_csv(OUT / "nested_selection.csv", index=False)
    files = ("per_match.csv", "model_summary.csv", "inner_epoch_history.csv", "nested_selection.csv")
    previous = json.loads((BASELINE_OUTPUT.parent / "run_manifest.json").read_text())
    manifest = dict(
        run_id=RUN_ID, fingerprints=fingerprint, grid=GRID, reused_folds=inherited,
        input_feature_sha256=previous["input_feature_sha256"], code_commit=previous["code_commit"],
        packages={name: version(name) for name in ("torch", "numpy", "pandas", "scipy", "pyarrow")},
        matches=MATCHES, learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY, max_epochs=MAX_EPOCHS,
        check_every=CHECK_EVERY, patience=PATIENCE, torch_seed=TORCH_SEED, common_preparation_s=preparation_s,
        output_sha256={name: digest(OUT / name) for name in files},
        nested_viterbi_f1=float(selected.viterbi_f1.mean()), nested_line_out_f1=float(selected.line_out_f1.mean()),
        total_grid_fold_s=sum(row["total_s"] for row in results),
    )
    (OUT / "run_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    (OUT / "validation.json").write_text(json.dumps(dict(
        passed=True, candidates=len(GRID), outer_matches=len(MATCHES), score_rows=len(table),
        nested_selected_rows=len(selected),
    ), indent=2) + "\n", encoding="utf-8")
    print(summary.round(4).to_string(index=False), flush=True)
    print(selected.round(4).to_string(index=False), flush=True)
    print(f"nested F1: {manifest['nested_viterbi_f1']:.4f} / {manifest['nested_line_out_f1']:.4f}", flush=True)
    return summary, selected


started = perf_counter()
train, _ = prepare_inputs()
evaluation = {match: validation_input(match, BASE) for match in MATCHES}
preparation_s = perf_counter() - started
fingerprint = {"notebook_sha256": digest(NOTEBOOK), "tuning_notebook_sha256": digest(tuning),
               "shared_notebook_sha256": digest(shared), "baseline_summary_sha256": digest(BASELINE_OUTPUT)}
checkpoint = OUT / "checkpoint.json"
if checkpoint.is_file():
    saved = json.loads(checkpoint.read_text())
    assert saved["fingerprints"] == fingerprint
    results = saved["results"]
else:
    results = []

completed = {(row["model"], row["match"]) for row in results}
inherited = 0
if PILOT.is_file():
    pilot_state = json.loads(PILOT.read_text())
    assert pilot_state["fingerprints"] == {
        "notebook_sha256": fingerprint["tuning_notebook_sha256"],
        "shared_notebook_sha256": fingerprint["shared_notebook_sha256"],
        "baseline_summary_sha256": fingerprint["baseline_summary_sha256"],
    }
    for fold in pilot_state["results"]:
        key = (fold["model"], fold["match"])
        assert key[0] in PILOT_CANDIDATES and key[1] in MATCHES
        inherited += 1
        if key not in completed:
            results.append(fold)
            completed.add(key)
    save_checkpoint(checkpoint, fingerprint, results)
    print(f"기존 튜닝 결과 {inherited}경기·조합 재사용", flush=True)

for name, held in product(GRID, MATCHES):
    if (name, held) not in completed:
        results.append(fit_fold(name, GRID[name], held, train, evaluation))
        save_checkpoint(checkpoint, fingerprint, results)
summary, nested_selection = report_grid(results, fingerprint, inherited, preparation_s)